# Machine Learning Model

In [5]:
# imports
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import joblib
from Encoder import encode, decode

In [8]:
model = joblib.load("/Users/hp/Downloads/Project_Week2_model/Project_Week2_model/ML Statmodel/price_model_statsmodels.pkl")

In [9]:
joblib.dump(model, "st_model.joblib", compress=3)

['st_model.joblib']

### Dataset

In [ ]:
# data read
data = pd.read_csv('/content/processed_car_sales_data_cleaning.csv')
data.head()

In [ ]:
# Columns division by type

# numerical columns
numerical = data[['Engine size', 'Year of manufacture', 'Mileage', 'Price']]

# catigorical columns
catigorical = data[['Manufacturer', 'Model','Fuel type']]

# change data types into categorey
data[catigorical.columns] = data[catigorical.columns].astype('category')

# one hot encoding for categorical data
data = encode(data)

In [ ]:
# Set plot styles
plt.style.use('default')
sns.set_palette("deep")

In [ ]:
# Feature selection
X = data.drop("Price", axis=1)  # Input features
y = data["Price"]  # Target: Price

In [ ]:
# Features Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

### Model

In [ ]:
# Model fit
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

##### Model Evaluation

In [ ]:
# Predictions Test
y_train_pred = rf.predict(X_train)
y_test_pred = rf.predict(X_test)

# Train metrics
print("Train R²:", r2_score(y_train, y_train_pred))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred)))

# Test metrics
print("Test R²:", r2_score(y_test, y_test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred)))

In [ ]:
# Corss Validation
scores = cross_val_score(rf, X, y, cv=5, scoring="r2")
print("Cross-Validation R² scores:", scores)
print("Mean CV R²:", scores.mean())

In [ ]:
# Learning Curve
train_sizes, train_scores, test_scores = learning_curve(
    rf, X, y, cv=5, scoring="r2", train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.plot(train_sizes, train_mean, label="Train R²")
plt.plot(train_sizes, test_mean, label="Test R²")
plt.xlabel("Training Samples")
plt.ylabel("R² Score")
plt.legend()
plt.show()

In [ ]:
# Model save
joblib.dump(rf, "price_model.joblib", compress = 5)
joblib.dump(scaler, os.path.join("scaler.pkl"))

In [ ]:
# Save metadata
features_list = X.columns.tolist()

model_metadata = {
    'features': features_list,
    'targets': ['Price'],
    'model_types': ['RandomForestRegressor'],
    'scaler': 'StandardScaler'
}

with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f)

### Prediction

In [ ]:
# Load the model
loaded_price_model = joblib.load('/content/price_model.joblib')

In [ ]:
# Create example data for prediction
example_data = pd.DataFrame([
    [1.8, 2015, 80000, 'Toyota', 'Yaris', 'Petrol'],
    [3.0, 2020, 20000, 'Porsche', '911', 'Petrol'],
    [2.0, 2018, 60000, 'VW', 'Golf', 'Hybrid'],
    [1.4, 2012, 120000, 'Ford', 'Fiesta', 'Petrol'],  # Ford Fiesta, Petrol
])

#encode
example_data = encode(example_data)

In [ ]:
# Make predictions
price_predictions = loaded_price_model.predict(example_data)

# Decode categorical columns
decoded_rows = decode(example_data)

for i in example_data.iterrows():
    decoded_rows.append({
        "Predicted Price": price_predictions[i]
    })

results = pd.DataFrame(decoded_rows)

# Display predictions
print("\nPredictions with readable labels:")
print(results)